# Edge AI Assignment: Neural Network from Scratch (Checkerboard Classification)

## Objective
Build a simple neural network **from scratch (no ML libraries)** to classify a 2D checkerboard pattern.

You will:
- Implement forward propagation
- Implement backpropagation
- Train a small neural network
- Visualize the decision boundary

---

## Dataset Concept
We generate (x, y) points and assign labels based on a checkerboard pattern.

This is a **non-linear classification problem** (like XOR, but extended to 2D).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Generate Checkerboard Data

In [ ]:
def generate_data(n_samples=5000, k=4):
    X = np.random.uniform(-1, 1, (n_samples, 2))

    # Checkerboard labels
    y = ((np.floor((X[:,0]+1)*k) + np.floor((X[:,1]+1)*k)) % 2)

    return X, y.reshape(-1, 1)

X, y = generate_data()

## Visualize Dataset

In [ ]:
plt.scatter(X[:,0], X[:,1], c=y.flatten())
plt.title("Checkerboard Dataset")
plt.show()

## Step 2: Build Neural Network

Architecture:
- Input: 2 neurons (x, y)
- Hidden: YOU DECIDE
- Output: 1 neuron (sigmoid)

## User Configuration

Set the number of hidden layers and neurons per layer below.
Each entry in `HIDDEN_LAYERS` is the neuron count for one hidden layer.

Examples:
- `[16]`          → 1 hidden layer with 16 neurons
- `[32, 16]`      → 2 hidden layers (32 then 16 neurons)
- `[64, 32, 16]`  → 3 hidden layers

In [ ]:
# ----- USER CONFIGURATION -----
HIDDEN_LAYERS = [64, 64]   # list of neuron counts, one per hidden layer
LEARNING_RATE = 0.1
EPOCHS        = 5000
LR_DECAY      = 0.001      # set to 0 to disable; lr shrinks as lr/(1 + decay*epoch)
# ------------------------------

print(f"Network: 2 → {' → '.join(str(n) for n in HIDDEN_LAYERS)} → 1")
print(f"Total hidden layers : {len(HIDDEN_LAYERS)}")
print(f"Learning rate       : {LEARNING_RATE}")
print(f"LR decay            : {LR_DECAY}")
print(f"Epochs              : {EPOCHS}")

In [ ]:
# CHANGE: He initialization (scale by sqrt(2/fan_in)) replaces the flat
# 0.1 scaling. He init keeps activation variance stable across ReLU layers,
# preventing gradients from vanishing or exploding as depth increases.
def init_params(layer_sizes):
    params = []
    for i in range(len(layer_sizes) - 1):
        fan_in = layer_sizes[i]
        W = np.random.randn(fan_in, layer_sizes[i + 1]) * np.sqrt(2.0 / fan_in)
        b = np.zeros((1, layer_sizes[i + 1]))
        params.append((W, b))
    return params

## Activation Functions

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(a):
    return a * (1 - a)

# CHANGE: added ReLU and its derivative for hidden layers.
# ReLU passes gradients through unchanged for positive inputs, which
# prevents the vanishing-gradient problem that sigmoid causes in deep nets.
def relu(x):
    return np.maximum(0, x)

def relu_derivative(a):
    return (a > 0).astype(float)

## Forward Pass

In [ ]:
# CHANGE: hidden layers now use ReLU; only the final output layer uses
# sigmoid (needed to produce a 0–1 probability for binary classification).
# The cache stores (A_in, Z, A_out, is_output) so backward knows which
# derivative to apply at each layer.
def forward(X, params):
    cache = []
    A = X
    for idx, (W, b) in enumerate(params):
        Z          = np.dot(A, W) + b
        is_output  = (idx == len(params) - 1)
        A_next     = sigmoid(Z) if is_output else relu(Z)
        cache.append((A, Z, A_next, is_output))
        A = A_next
    return cache

## Loss Function

In [ ]:
# CHANGE: implemented binary cross-entropy loss.
# A small epsilon (1e-8) prevents log(0) when predictions are exactly 0 or 1.
def compute_loss(y_true, y_pred):
    eps  = 1e-8
    loss = -np.mean(
        y_true * np.log(y_pred + eps) +
        (1 - y_true) * np.log(1 - y_pred + eps)
    )
    return loss

## Backpropagation

X is the input, y error W1, b1, W2, b2 are the original weights

In [ ]:
# CHANGE: backward now uses relu_derivative for hidden layers instead of
# sigmoid_derivative. The output delta is still (A_out - y) because that
# simplification holds regardless of the hidden activation choice.
def backward(X, y, params, cache):
    m     = X.shape[0]
    grads = [None] * len(params)

    A_out = cache[-1][2]
    delta = A_out - y          # combined cross-entropy + sigmoid derivative

    for i in reversed(range(len(params))):
        A_prev, Z, A_cur, is_output = cache[i]

        dW = np.dot(A_prev.T, delta) / m
        db = np.mean(delta, axis=0, keepdims=True)
        grads[i] = (dW, db)

        if i > 0:
            W     = params[i][0]
            A_prev_layer = cache[i - 1][2]
            delta = np.dot(delta, W.T) * relu_derivative(A_prev_layer)

    return grads

## Training Loop

   Train the network using gradient descent.

    Parameters
    ----------
    X           : ndarray (n × 2)  – training inputs
    y           : ndarray (n × 1)  – training labels
    hidden_size : int              – number of hidden neurons
                                    CHANGE: default raised from 8 → 32 so
                                    the network has enough capacity to fit
                                    the complex checkerboard pattern.
    lr          : float            – learning rate (step size for updates)
    epochs      : int              – how many full passes over the data
                                    CHANGE: default raised from 1000 → 5000
                                    for better convergence.

    Returns
    -------
    W1, b1, W2, b2 – the trained parameters

    How gradient descent works
    --------------------------
    On each epoch:
      1. Forward pass  → get predictions
      2. Loss          → measure how wrong we are
      3. Backward pass → compute how each weight contributed to the error
      4. Update        → nudge every weight *opposite* to its gradient
                         (moving downhill on the loss surface)

         W  ←  W  −  lr · dL/dW

    The learning rate `lr` controls the step size.  Too large → overshoots;
    too small → learns very slowly.

In [ ]:
# CHANGE: train now accepts lr_decay for learning rate annealing.
# The effective rate shrinks as lr / (1 + decay * epoch), which lets the
# network take large steps early and fine-tune at the end.
def train(X, y, hidden_layers=None, lr=0.1, epochs=5000, lr_decay=0.0):
    if hidden_layers is None:
        hidden_layers = [64, 64]

    layer_sizes = [2] + hidden_layers + [1]
    params      = init_params(layer_sizes)

    for epoch in range(epochs):
        cache        = forward(X, params)
        final_output = cache[-1][2]

        loss  = compute_loss(y, final_output)
        grads = backward(X, y, params, cache)

        lr_eff = lr / (1 + lr_decay * epoch)
        params = [
            (W - lr_eff * dW, b - lr_eff * db)
            for (W, b), (dW, db) in zip(params, grads)
        ]

        if epoch % 500 == 0:
            print(f"Epoch {epoch:>5}, Loss: {loss:.4f}, LR: {lr_eff:.5f}")

    return params

## Visualize Decision Boundary

In [ ]:
# CHANGE: plot_decision_boundary now accepts a params list (not a
# fixed four-variable model) and calls forward to get predictions,
# completing the TODO left in the original notebook.
def plot_decision_boundary(X, y, params):
    xx, yy = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
    grid   = np.c_[xx.ravel(), yy.ravel()]

    cache = forward(grid, params)
    Z     = cache[-1][2].reshape(xx.shape)

    plt.contourf(xx, yy, Z, levels=50, alpha=0.6, cmap='RdBu')
    plt.scatter(X[:, 0], X[:, 1], c=y.flatten(), s=2, cmap='RdBu')
    plt.title(f"Decision Boundary — hidden layers: {[p[0].shape[1] for p in params[:-1]]}")
    plt.colorbar(label='P(class=1)')
    plt.show()

## Run Training

In [ ]:
# Uses the HIDDEN_LAYERS / LEARNING_RATE / EPOCHS / LR_DECAY set in the
# User Configuration cell above.
params = train(X, y,
               hidden_layers=HIDDEN_LAYERS,
               lr=LEARNING_RATE,
               epochs=EPOCHS,
               lr_decay=LR_DECAY)

plot_decision_boundary(X, y, params)

---

## ⭐ Extra Credit ( For Concept And Code, 20% add on )

### Circle Classification Problem

Instead of a checkerboard, classify points based on whether they lie inside a circle:

- Input: (x, y)
- Output: 1 if inside circle, else 0

Decision boundary:
x² + y² < r²

### Questions:
1. Why is this non-linear?
2. Would a single-layer perceptron work?
3. How would the decision boundary differ from the checkerboard?

---